In [ ]:
import os
import json
from pathlib import Path
from ultralytics import YOLO

model = YOLO("data/yolov8n.pt") 
model.to('cuda')
output_path = 'data/yolo'
folder_path = 'data/keyframes'

if not os.path.isdir(output_path):
   os.makedirs(output_path)

for keyframes in os.listdir(folder_path):
    path_keyframes = os.path.join(folder_path, keyframes)
    output_path_keyframes = os.path.join(output_path, keyframes)
    if not os.path.isdir(output_path_keyframes):
        os.makedirs(output_path_keyframes)

    if os.path.isdir(path_keyframes):
        for subkeyframes in os.listdir(path_keyframes):
            path_subkeyframes = os.path.join(path_keyframes, subkeyframes)
            output_path_subkeyframes = os.path.join(output_path_keyframes, subkeyframes)
            if not os.path.isdir(output_path_subkeyframes):
                os.makedirs(output_path_subkeyframes)
            
            if os.path.isdir(path_subkeyframes):
                for path in Path(path_subkeyframes).glob('*.jpg'):
                    base, extension = os.path.splitext(path.name)
                    jsonFileName = subkeyframes + "-" + base
                    jsonFilePath = output_path_subkeyframes + "/" + jsonFileName + ".json"
                    
                    results = model.predict(path)
                    result = results[0]
                    objects = {}
                    i = 0
                    for box in result.boxes:
                        class_id = result.names[box.cls[0].item()]
                        conf = round(box.conf[0].item(), 2)

                        if (conf > 0.5):
                            if class_id in objects:
                                objects[class_id] += 1
                            else:
                                objects[class_id] = 1
                            i += 1
                    json_string = json.dumps(objects, indent=4)
                    with open(jsonFilePath, 'w') as f:
                        f.write(json_string)